# Round-wire CLN: elements, expansion point, and circuit export

Install the solver dependencies with `pip install -e ".[solver,test]"` from the repository root. This example runs NGSolve and needs only open-source tools.

All electromagnetic impedances and ladder elements below are **per metre**. The Laplace variable `s` is real for the expansion-point check; sinusoidal response uses `s = 1j * 2*pi*f`. A positive real `s0` is not an AC frequency. The first resistance is `Z(s0)`; only a ladder expanded at `s0=0` has `R[0]=Z(0)`.

The source now explicitly selects the direct backend for positive shifts. Saved outputs predate this source edit and have not been re-executed. The checks compare the finite-element ladder with an independent direct solve and the analytic round-wire solution. Tolerances apply to this mesh and order, not every geometry.

In [1]:
from pathlib import Path
import sys
# Run from the repository root or docs/, or install CLN-mcp first.
for root in (Path.cwd(), Path.cwd().parent):
    if (root / "src" / "cln").is_dir():
        sys.path.insert(0, str(root / "src"))
        break
import numpy as np
from cln import (TOmegaCLN, cylinder_mesh, ladder_impedance,
                 reference_impedance, wire_internal_impedance,
                 wire_cln_s0_zero, ladder_netlist)
a, h, sigma, mu = 0.01, 0.01, 1e6, 4e-7*np.pi
R_exact = {n: wire_cln_s0_zero(n, a, sigma, mu) for n in (0, 2, 4, 6)}
L_exact = {n: wire_cln_s0_zero(n, a, sigma, mu) for n in (1, 3, 5)}
assert np.isclose(wire_internal_impedance(0, a, sigma, mu), R_exact[0])
print("DC resistance [ohm/m]:", R_exact[0])


DC resistance [ohm/m]: 0.0031830988618379067


In [2]:
mesh = cylinder_mesh(a, h, maxh=a/2, maxh_conductor=a/10)
s0 = 2*np.pi*1e5
R, L, _ = TOmegaCLN(mesh, sigma, mu, a, h, s0=s0, order=2, solver="direct").run(nmax=4)
for s in (s0, 1.5*s0):
    reduced = ladder_impedance(R, L, s, s0=s0)
    direct = reference_impedance(mesh, sigma, mu, a, h, s, order=2)
    exact = wire_internal_impedance(s, a, sigma, mu)
    rel_reduction = abs(reduced/direct - 1)
    rel_mesh = abs(direct/exact - 1)
    assert rel_reduction < 2e-4
    assert rel_mesh < 3e-3
    print(f"s={s:.6g}: reduced/direct error={rel_reduction:.3e}; direct/analytic error={rel_mesh:.3e}")


s=628319: reduced/direct error=4.441e-16; direct/analytic error=6.106e-04


s=942478: reduced/direct error=2.016e-05; direct/analytic error=1.288e-03


## Export a finite segment

Multiply the per-metre values by the segment length when exporting. `style="rl"` explicitly sets `Rser=0` so LTspice does not add its default winding resistance. A shifted ladder includes negative resistors implementing `(s-s0)L`; the approximation should be checked over the intended operating range.

In [3]:
netlist = ladder_netlist(R, L, s0=s0, length=h, style="rl")
assert "Rser=0" in netlist
print(netlist)


* CLN: Cauer ladder, s0 = 628318.531 1/s, nmax = 4, length = 0.01 m
RCLN0 p CLN_0 0.000150232175
LCLN1 CLN_0 CLN_1m 1.12185389e-10 Rser=0
RCLN1neg CLN_1m 0 -7.04881589e-05
RCLN2 CLN_0 CLN_2 0.000289240811
LCLN3 CLN_2 CLN_3m 1.16219237e-10 Rser=0
RCLN3neg CLN_3m 0 -7.30227001e-05
RCLN4 CLN_2 0 0.000319911446

